# 02-19 Local nnU-Net PHE with pseudo-ICH prior control

Scientific control for `02_14b`: use the same PHE-SICH split and nnU-Net PHE pipeline, but replace the external ICH teacher with pseudo-ICH maps generated directly from the PHE-SICH CT images.

Local setup: read `nnUNet-master` and `PHE-SICH-CT-IDS` from this workspace, write all outputs locally, and train with vanilla nnU-Net `nnUNetTrainer_250epochs`.

Fairness rule: pseudo-ICH priors are generated from CT intensities only. Manual PHE masks are used only as PHE labels/evaluation targets and optional QC overlays, not to generate prior channels.


In [ ]:
from pathlib import Path
import os
import sys
import re
import json
import shutil
import time
import gc

import numpy as np
import pandas as pd

# Local-only setup for this notebook. Keep Kaggle disabled so paths do not drift.
IS_KAGGLE = False
KAGGLE_INPUT = None
WORK_ROOT = Path.cwd().resolve()
LOCAL_ROOT = WORK_ROOT
PROJECT_ROOT = WORK_ROOT

# Local manual paths. Edit these only if your workspace layout changes.
USER_NNUNET_REPO = PROJECT_ROOT / 'nnUNet-master'
USER_PHE_ROOT = PROJECT_ROOT / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT'
USER_SPLIT_CSV = None    # leave None to use the embedded fair split IDs

VAL_SCAN_IDS = {'0013', '0014', '0060', '0078', '0080', '0115', '0119', '0130', '0138', '0160'}
TEST_SCAN_IDS = {'0002', '0029', '0033', '0051', '0061', '0084', '0095', '0096', '0113', '0116', '0167'}

# 02_19 is a no-teacher control. It does not load 02_13/02_13b checkpoints.
PSEUDO_ICH_SOURCE = 'ct_threshold_no_gt'
PSEUDO_ICH_POLICY = (
    'Pseudo-ICH prior is generated from PHE-SICH CT intensities only; '
    'no true ICH labels and no PHE masks are used for prior generation.'
)

EXPERIMENTS = {
    'phe_pseudo_binary_prior': {
        'dataset_id': 190,
        'prior_channels': ['ich_binary'],
        'description': 'CT + pseudo hard ICH mask from CT thresholding',
    },
    'phe_pseudo_probability_prior': {
        'dataset_id': 191,
        'prior_channels': ['ich_probability'],
        'description': 'CT + pseudo soft ICH probability from CT thresholding',
    },
    'phe_pseudo_distance_prior': {
        'dataset_id': 192,
        'prior_channels': ['ich_distance'],
        'description': 'CT + distance-to-pseudo-ICH proximity map',
    },
    'phe_pseudo_prob_distance_prior': {
        'dataset_id': 193,
        'prior_channels': ['ich_probability', 'ich_distance'],
        'description': 'CT + pseudo ICH probability + distance-to-pseudo-ICH proximity',
    },
    'phe_pseudo_prob_distance_dilated_prior': {
        'dataset_id': 194,
        'prior_channels': ['ich_probability', 'ich_distance', 'ich_dilated_roi'],
        'description': 'CT + pseudo ICH probability + distance + dilated ROI',
    },
}

EXPERIMENT_ID = 'phe_pseudo_prob_distance_prior'
assert EXPERIMENT_ID in EXPERIMENTS, f'Unknown EXPERIMENT_ID: {EXPERIMENT_ID}'
EXPERIMENT = EXPERIMENTS[EXPERIMENT_ID]
PRIOR_CHANNELS = list(EXPERIMENT['prior_channels'])

OUTPUT_BASE = WORK_ROOT / 'outputs_02_19_nnunet_phe_pseudo_ich_prior_control'
OUTPUT_ROOT = OUTPUT_BASE / EXPERIMENT_ID
PSEUDO_ICH_OUTPUT_ROOT = OUTPUT_BASE / 'pseudo_ich_outputs' / 'phe_sich_ct_threshold_prior'
NNUNET_BASE = OUTPUT_ROOT / 'nnunet_workdir'
NNUNET_RAW = NNUNET_BASE / 'nnUNet_raw'
NNUNET_PREPROCESSED = NNUNET_BASE / 'nnUNet_preprocessed'
# Keep results short on Windows to avoid long path failures during checkpointing.
NNUNET_RESULTS = WORK_ROOT / 'o19_results' / EXPERIMENT_ID

DATASET_ID = int(EXPERIMENT['dataset_id'])
DATASET_NAME = f'Dataset{DATASET_ID:03d}_PHESICH_PHE_{EXPERIMENT_ID}'
DATASET_DIR = NNUNET_RAW / DATASET_NAME

ICH_PRIOR_PROB_DIR = PSEUDO_ICH_OUTPUT_ROOT / 'probability_maps'
ICH_PRIOR_MASK_DIR = PSEUDO_ICH_OUTPUT_ROOT / 'binary_masks'
ICH_DISTANCE_PRIOR_DIR = PSEUDO_ICH_OUTPUT_ROOT / 'distance_maps'
ICH_DILATED_ROI_DIR = PSEUDO_ICH_OUTPUT_ROOT / 'dilated_roi'
PRIOR_QC_DIR = PSEUDO_ICH_OUTPUT_ROOT / 'visualizations'


def strip_nii_suffix(path_like):
    name = Path(str(path_like)).name
    if name.endswith('.nii.gz'):
        return name[:-7]
    if name.endswith('.nii'):
        return name[:-4]
    return Path(name).stem


def scan_id_from_value(value):
    text = strip_nii_suffix(value).strip()
    if re.fullmatch(r'\d+\.0+', text):
        text = text.split('.')[0]
    m = re.search(r'(\d{4})$', text)
    if not m:
        m = re.search(r'(\d+)$', text)
    if not m:
        raise ValueError(f'Cannot parse scan id from {value}')
    return m.group(1).zfill(4)


def make_case_id(scan_id):
    scan_id = str(scan_id).strip()
    if scan_id.lower().startswith('phe_'):
        return scan_id
    if scan_id.isdigit():
        return f'phe_{int(scan_id):04d}'
    safe = ''.join(ch if ch.isalnum() else '_' for ch in scan_id).strip('_')
    return f'phe_{safe}'


def split_from_scan_id(scan_id):
    scan_id4 = scan_id_from_value(scan_id)
    if scan_id4 in TEST_SCAN_IDS:
        return 'test'
    if scan_id4 in VAL_SCAN_IDS:
        return 'val'
    return 'train'


def has_nifti_pair_root(path_like):
    root = Path(path_like)
    if not (root / 'set').exists() or not (root / 'label').exists():
        return False
    image_count = len(list((root / 'set').glob('*.nii'))) + len(list((root / 'set').glob('*.nii.gz')))
    mask_count = len(list((root / 'label').glob('*.nii'))) + len(list((root / 'label').glob('*.nii.gz')))
    return image_count > 0 and mask_count > 0


def find_nnunet_repo():
    if USER_NNUNET_REPO is not None:
        repo = Path(USER_NNUNET_REPO)
        if (repo / 'nnunetv2' / '__init__.py').exists():
            return repo
        raise FileNotFoundError(f'USER_NNUNET_REPO is not an nnU-Net repo: {repo}')
    candidates = [LOCAL_ROOT / 'nnUNet-master', LOCAL_ROOT]
    if IS_KAGGLE:
        candidates.extend(KAGGLE_INPUT.iterdir())
    for base in candidates:
        if (base / 'nnunetv2' / '__init__.py').exists():
            return base
        if (base / 'nnUNet-master' / 'nnunetv2' / '__init__.py').exists():
            return base / 'nnUNet-master'
    search_roots = [KAGGLE_INPUT] if IS_KAGGLE else [LOCAL_ROOT]
    for root in search_roots:
        for init_file in root.rglob('nnunetv2/__init__.py'):
            return init_file.parents[1]
    raise FileNotFoundError('Could not find local nnU-Net repo. Set USER_NNUNET_REPO to your nnUNet-master folder.')


def find_phe_root():
    if USER_PHE_ROOT is not None:
        root = Path(USER_PHE_ROOT)
        if has_nifti_pair_root(root):
            return root
        raise FileNotFoundError(f'USER_PHE_ROOT does not contain set/label NIfTI pairs: {root}')
    candidates = [
        LOCAL_ROOT / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
        LOCAL_ROOT / 'SubdatasetA_NIFIT' / 'NIFIT',
    ]
    if IS_KAGGLE:
        for base in KAGGLE_INPUT.iterdir():
            candidates.extend([
                base / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
                base / 'SubdatasetA_NIFIT' / 'NIFIT',
                base / 'NIFIT',
                base,
            ])
    for root in candidates:
        if has_nifti_pair_root(root):
            return root
    search_roots = [KAGGLE_INPUT] if IS_KAGGLE else [LOCAL_ROOT]
    found = []
    for root in search_roots:
        for set_dir in root.rglob('set'):
            candidate = set_dir.parent
            if has_nifti_pair_root(candidate):
                found.append(candidate)
    if found:
        found = sorted(found, key=lambda x: (('SubdatasetA_NIFIT' not in str(x)), len(str(x))))
        return found[0]
    raise FileNotFoundError('Could not find PHE NIfTI root with set/ and label/. Set USER_PHE_ROOT manually.')


def find_split_csv():
    if USER_SPLIT_CSV is not None and Path(USER_SPLIT_CSV).exists():
        return Path(USER_SPLIT_CSV)
    names = {'3dff_iph_phe_patient_split.csv', 'phe_split.csv', 'patient_split.csv'}
    candidates = [
        LOCAL_ROOT / 'outputs_02_10_pese_guided_3dff_refined_pseudo_iph_phe_25d_segmentation' / 'manifests' / '3dff_iph_phe_patient_split.csv'
    ]
    if IS_KAGGLE:
        for root in KAGGLE_INPUT.iterdir():
            for csv_path in root.rglob('*.csv'):
                if csv_path.name in names:
                    candidates.append(csv_path)
    for csv_path in candidates:
        if csv_path.exists():
            return csv_path
    return None


def build_nifti_index(folder):
    files = sorted(list(Path(folder).glob('*.nii')) + list(Path(folder).glob('*.nii.gz')))
    out = {}
    for file_path in files:
        try:
            out[scan_id_from_value(file_path)] = file_path
        except ValueError:
            pass
    return out


def find_nii_by_scan(folder, scan_id):
    scan_id4 = scan_id_from_value(scan_id)
    direct_candidates = [
        Path(folder) / f'{scan_id4}.nii.gz',
        Path(folder) / f'{scan_id4}.nii',
        Path(folder) / f'{int(scan_id4)}.nii.gz',
        Path(folder) / f'{int(scan_id4)}.nii',
    ]
    for candidate in direct_candidates:
        if candidate.exists():
            return candidate
    indexed = build_nifti_index(folder)
    if scan_id4 in indexed:
        return indexed[scan_id4]
    raise FileNotFoundError(f'Missing NIfTI for scan_id={scan_id4} in {folder}')


NNUNET_REPO = find_nnunet_repo()
PHE_ROOT = find_phe_root()
PHE_IMAGE_DIR = PHE_ROOT / 'set'
PHE_MASK_DIR = PHE_ROOT / 'label'
SPLIT_CSV = find_split_csv()

for path in [
    OUTPUT_BASE, OUTPUT_ROOT, PSEUDO_ICH_OUTPUT_ROOT,
    NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS,
    ICH_PRIOR_PROB_DIR, ICH_PRIOR_MASK_DIR, ICH_DISTANCE_PRIOR_DIR,
    ICH_DILATED_ROI_DIR, PRIOR_QC_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw'] = str(NNUNET_RAW)
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED)
os.environ['nnUNet_results'] = str(NNUNET_RESULTS)
os.environ['nnUNet_n_proc_DA'] = '0'
os.environ['nnUNet_compile'] = 'False'

if str(NNUNET_REPO) not in sys.path:
    sys.path.insert(0, str(NNUNET_REPO))

print('IS_KAGGLE            :', IS_KAGGLE)
print('WORK_ROOT            :', WORK_ROOT)
print('Experiment           :', EXPERIMENT_ID, '-', EXPERIMENT['description'])
print('Prior channels       :', PRIOR_CHANNELS)
print('Pseudo ICH source    :', PSEUDO_ICH_SOURCE)
print('nnU-Net repo         :', NNUNET_REPO)
print('PHE root             :', PHE_ROOT)
print('PHE image dir        :', PHE_IMAGE_DIR)
print('PHE mask dir         :', PHE_MASK_DIR)
print('Split CSV            :', SPLIT_CSV if SPLIT_CSV is not None else 'not found, using embedded split IDs')
print('Images               :', len(list(PHE_IMAGE_DIR.glob('*.nii*'))))
print('Masks                :', len(list(PHE_MASK_DIR.glob('*.nii*'))))
print('Output root          :', OUTPUT_ROOT)
print('Pseudo ICH output    :', PSEUDO_ICH_OUTPUT_ROOT)
print('Dataset              :', DATASET_NAME)
print('nnUNet_raw           :', os.environ['nnUNet_raw'])
print('nnUNet_preprocessed  :', os.environ['nnUNet_preprocessed'])
print('nnUNet_results       :', os.environ['nnUNet_results'])


## 0. Local path setup

This notebook expects local workspace folders:

- `nnUNet-master/` containing `nnunetv2/__init__.py`
- `PHE-SICH-CT-IDS/SubdatasetA_NIFIT/NIFIT/set` and `label`

If `USER_SPLIT_CSV` is `None`, the notebook uses the embedded fair val/test IDs used by the recent PHE-SICH nnU-Net baselines.


In [ ]:
AUTO_INSTALL_MISSING_DEPS = False
INSTALL_NNUNET_EDITABLE = False


def ensure_import(import_name, pip_name=None):
    import importlib
    import subprocess
    pip_name = pip_name or import_name
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError as exc:
        if not AUTO_INSTALL_MISSING_DEPS:
            raise ModuleNotFoundError(
                f'Missing {import_name}. Use the Python 3.11 kernel/environment that already runs nnU-Net, '
                f'or set AUTO_INSTALL_MISSING_DEPS=True in this cell to install {pip_name}.'
            ) from exc
        cmd = [sys.executable, '-m', 'pip', 'install', pip_name]
        print('Missing', import_name, '-> running:', ' '.join(cmd))
        subprocess.check_call(cmd)
        return importlib.import_module(import_name)


if INSTALL_NNUNET_EDITABLE:
    import subprocess
    cmd = [sys.executable, '-m', 'pip', 'install', '-e', str(NNUNET_REPO)]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

ensure_import('SimpleITK')
ensure_import('scipy')
ensure_import('batchgenerators')
ensure_import('batchgeneratorsv2')
ensure_import('acvl_utils', 'acvl-utils')
ensure_import('dynamic_network_architectures', 'dynamic-network-architectures')
ensure_import('blosc2')
ensure_import('yacs')
ensure_import('nnunetv2')

import SimpleITK as sitk
from scipy import ndimage as ndi
import nnunetv2
print('SimpleITK OK')
print('scipy OK')
print('nnunetv2 import OK:', nnunetv2.__file__)


## 1. Build PHE-SICH split manifest

If `USER_SPLIT_CSV` is present, its split labels are reused but image/mask paths are remapped to the local PHE-SICH dataset. If no CSV is present, the notebook uses the same embedded split IDs as the recent PHE-SICH nnU-Net baselines: train 99, val 10, test 11.


In [ ]:
image_by_scan = build_nifti_index(PHE_IMAGE_DIR)
mask_by_scan = build_nifti_index(PHE_MASK_DIR)
assert image_by_scan, f'No NIfTI images found in {PHE_IMAGE_DIR}'
assert mask_by_scan, f'No NIfTI masks found in {PHE_MASK_DIR}'

rows = []
if SPLIT_CSV is not None:
    raw_split_df = pd.read_csv(SPLIT_CSV, dtype={'scan_id': str, 'patient_id': str})
    raw_split_df.columns = [str(c).strip() for c in raw_split_df.columns]
    for raw_row in raw_split_df.to_dict('records'):
        scan_source = raw_row.get('scan_id', None)
        if scan_source is None or pd.isna(scan_source) or str(scan_source).strip() == '':
            scan_source = raw_row.get('patient_id', None)
        if scan_source is None or pd.isna(scan_source) or str(scan_source).strip() == '':
            scan_source = raw_row.get('img_path', raw_row.get('image_path', ''))
        scan_id = scan_id_from_value(scan_source)
        split_value = str(raw_row.get('split', split_from_scan_id(scan_id))).strip().lower()
        if split_value not in {'train', 'val', 'test'}:
            split_value = split_from_scan_id(scan_id)

        img_path = Path(str(raw_row.get('img_path', raw_row.get('image_path', ''))))
        mask_path = Path(str(raw_row.get('phe_mask_path', raw_row.get('label_path', raw_row.get('mask_path', '')))))
        if not img_path.exists():
            img_path = image_by_scan.get(scan_id) or find_nii_by_scan(PHE_IMAGE_DIR, scan_id)
        if not mask_path.exists():
            mask_path = mask_by_scan.get(scan_id) or find_nii_by_scan(PHE_MASK_DIR, scan_id)
        rows.append({
            'scan_id': scan_id,
            'case_id': make_case_id(scan_id),
            'split': split_value,
            'img_path': str(img_path),
            'phe_mask_path': str(mask_path),
        })
else:
    for scan_id, img_path in sorted(image_by_scan.items()):
        mask_path = mask_by_scan.get(scan_id)
        if mask_path is None:
            print('WARNING: missing mask for image scan_id:', scan_id, img_path)
            continue
        rows.append({
            'scan_id': scan_id,
            'case_id': make_case_id(scan_id),
            'split': split_from_scan_id(scan_id),
            'img_path': str(img_path),
            'phe_mask_path': str(mask_path),
        })

split_df = pd.DataFrame(rows).drop_duplicates('case_id', keep='first').sort_values(['split', 'scan_id']).reset_index(drop=True)
required_cols = {'case_id', 'split', 'img_path', 'phe_mask_path'}
missing_cols = required_cols - set(split_df.columns)
assert not missing_cols, f'Missing columns in split manifest: {missing_cols}'
assert split_df['case_id'].is_unique, 'case_id must be unique.'
assert split_df['img_path'].map(lambda x: Path(x).exists()).all(), 'Some image paths are missing.'
assert split_df['phe_mask_path'].map(lambda x: Path(x).exists()).all(), 'Some PHE mask paths are missing.'

manifest_csv = OUTPUT_ROOT / '02_19_local_phe_sich_split_manifest.csv'
split_df.to_csv(manifest_csv, index=False)

print('Total cases:', len(split_df))
print('Manifest:', manifest_csv)
display(split_df.groupby('split').size().rename('cases').reset_index())
display(split_df.head())


## 2. Generate pseudo-ICH priors directly on PHE-SICH CT

This control replaces the `02_13b` teacher with a deterministic CT-only pseudo-labeler. It intentionally avoids true ICH labels and avoids PHE masks when creating prior channels, so the comparison with `02_14b` stays fair.


In [ ]:
RUN_PSEUDO_ICH_PRIOR_GENERATION = True
OVERWRITE_PSEUDO_ICH_PRIORS = False

# HU-like threshold policy for acute hematoma candidates. Keep these fixed when comparing with 02_14b.
PSEUDO_ICH_SOFT_LOW_HU = 35.0
PSEUDO_ICH_CORE_LOW_HU = 45.0
PSEUDO_ICH_CORE_HIGH_HU = 85.0
PSEUDO_ICH_SOFT_HIGH_HU = 110.0
PSEUDO_ICH_MIN_VOLUME_ML = 0.20
PSEUDO_ICH_MAX_VOLUME_ML = 120.0
PSEUDO_ICH_KEEP_LARGEST_COMPONENTS = 3
PSEUDO_ICH_OPENING_RADIUS_MM = 1.0
PSEUDO_ICH_CLOSING_RADIUS_MM = 1.5
PSEUDO_ICH_SOFT_SUPPORT_RADIUS_MM = 2.0
DISTANCE_SCALE_MM = 10.0
DILATED_ROI_RADIUS_MM = 20.0

for p in [ICH_PRIOR_PROB_DIR, ICH_PRIOR_MASK_DIR, ICH_DISTANCE_PRIOR_DIR, ICH_DILATED_ROI_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if OVERWRITE_PSEUDO_ICH_PRIORS:
    for p in [ICH_PRIOR_PROB_DIR, ICH_PRIOR_MASK_DIR, ICH_DISTANCE_PRIOR_DIR, ICH_DILATED_ROI_DIR]:
        for f in p.glob('*.nii.gz'):
            f.unlink()


def write_array_like_reference(arr, reference_path: Path, out_path: Path, dtype=np.float32):
    ref = sitk.ReadImage(str(reference_path))
    out = sitk.GetImageFromArray(np.asarray(arr).astype(dtype))
    if out.GetSize() != ref.GetSize():
        raise ValueError(f'Size mismatch for {out_path}: {out.GetSize()} vs ref {ref.GetSize()}')
    out.CopyInformation(ref)
    sitk.WriteImage(out, str(out_path))


def make_ball_structure(spacing_xyz, radius_mm):
    radius_mm = float(radius_mm)
    if radius_mm <= 0:
        return np.ones((1, 1, 1), dtype=bool)
    spacing_zyx = np.asarray(spacing_xyz[::-1], dtype=float)
    radii = np.maximum(1, np.ceil(radius_mm / np.maximum(spacing_zyx, 1e-6)).astype(int))
    zz, yy, xx = np.ogrid[-radii[0]:radii[0] + 1, -radii[1]:radii[1] + 1, -radii[2]:radii[2] + 1]
    dist_mm = np.sqrt((zz * spacing_zyx[0]) ** 2 + (yy * spacing_zyx[1]) ** 2 + (xx * spacing_zyx[2]) ** 2)
    return dist_mm <= radius_mm


def trapezoid_probability(ct_arr, soft_low, core_low, core_high, soft_high):
    ct = np.asarray(ct_arr, dtype=np.float32)
    prob = np.zeros(ct.shape, dtype=np.float32)
    rising = (ct >= soft_low) & (ct < core_low)
    core = (ct >= core_low) & (ct <= core_high)
    falling = (ct > core_high) & (ct <= soft_high)
    prob[rising] = (ct[rising] - soft_low) / max(core_low - soft_low, 1e-6)
    prob[core] = 1.0
    prob[falling] = (soft_high - ct[falling]) / max(soft_high - core_high, 1e-6)
    return np.clip(prob, 0.0, 1.0).astype(np.float32)


def filter_pseudo_ich_components(candidate, spacing_xyz, min_volume_ml, max_volume_ml, keep_largest):
    candidate = np.asarray(candidate, dtype=bool)
    if not candidate.any():
        return np.zeros(candidate.shape, dtype=np.uint8), []
    labeled, num = ndi.label(candidate)
    voxel_ml = float(np.prod(np.asarray(spacing_xyz, dtype=float)) / 1000.0)
    components = []
    for comp_id in range(1, num + 1):
        comp = labeled == comp_id
        voxels = int(comp.sum())
        volume_ml = voxels * voxel_ml
        if volume_ml < float(min_volume_ml) or volume_ml > float(max_volume_ml):
            continue
        components.append((comp_id, voxels, volume_ml))
    components = sorted(components, key=lambda x: x[1], reverse=True)[:int(keep_largest)]
    out = np.zeros(candidate.shape, dtype=bool)
    for comp_id, _, _ in components:
        out |= labeled == comp_id
    return out.astype(np.uint8), [
        {'component_id': int(comp_id), 'voxels': int(voxels), 'volume_ml': float(volume_ml)}
        for comp_id, voxels, volume_ml in components
    ]


def make_distance_prior(binary_ich, spacing_xyz, scale_mm=10.0):
    binary_ich = np.asarray(binary_ich).astype(bool)
    if not binary_ich.any():
        return np.zeros(binary_ich.shape, dtype=np.float32)
    spacing_zyx = tuple(float(x) for x in spacing_xyz[::-1])
    dist_mm = ndi.distance_transform_edt(~binary_ich, sampling=spacing_zyx)
    return np.exp(-dist_mm / float(scale_mm)).astype(np.float32)


def make_dilated_roi(binary_ich, spacing_xyz, radius_mm=20.0):
    binary_ich = np.asarray(binary_ich).astype(bool)
    if not binary_ich.any():
        return np.zeros(binary_ich.shape, dtype=np.uint8)
    spacing_zyx = tuple(float(x) for x in spacing_xyz[::-1])
    dist_mm = ndi.distance_transform_edt(~binary_ich, sampling=spacing_zyx)
    return (dist_mm <= float(radius_mm)).astype(np.uint8)


def generate_pseudo_ich_from_ct(ct_arr, spacing_xyz):
    ct = np.nan_to_num(np.asarray(ct_arr, dtype=np.float32), nan=-1024.0, posinf=3071.0, neginf=-1024.0)
    intensity_prob = trapezoid_probability(
        ct,
        PSEUDO_ICH_SOFT_LOW_HU,
        PSEUDO_ICH_CORE_LOW_HU,
        PSEUDO_ICH_CORE_HIGH_HU,
        PSEUDO_ICH_SOFT_HIGH_HU,
    )
    candidate = intensity_prob >= 0.5
    open_structure = make_ball_structure(spacing_xyz, PSEUDO_ICH_OPENING_RADIUS_MM)
    close_structure = make_ball_structure(spacing_xyz, PSEUDO_ICH_CLOSING_RADIUS_MM)
    if candidate.any():
        candidate = ndi.binary_opening(candidate, structure=open_structure)
        candidate = ndi.binary_closing(candidate, structure=close_structure)
    binary, components = filter_pseudo_ich_components(
        candidate,
        spacing_xyz,
        min_volume_ml=PSEUDO_ICH_MIN_VOLUME_ML,
        max_volume_ml=PSEUDO_ICH_MAX_VOLUME_ML,
        keep_largest=PSEUDO_ICH_KEEP_LARGEST_COMPONENTS,
    )
    support = make_dilated_roi(binary, spacing_xyz, radius_mm=PSEUDO_ICH_SOFT_SUPPORT_RADIUS_MM).astype(bool)
    probability = np.where(support, intensity_prob, 0.0).astype(np.float32)
    probability[binary.astype(bool)] = np.maximum(probability[binary.astype(bool)], 0.75)
    return probability, binary.astype(np.uint8), components


def prior_file_ready(path: Path):
    return path.exists() and path.stat().st_size > 0


expected_prior_groups = []
for row in split_df.itertuples(index=False):
    case_id = str(row.case_id)
    expected_prior_groups.append({
        'case_id': case_id,
        'probability_path': ICH_PRIOR_PROB_DIR / f'{case_id}.nii.gz',
        'binary_path': ICH_PRIOR_MASK_DIR / f'{case_id}.nii.gz',
        'distance_path': ICH_DISTANCE_PRIOR_DIR / f'{case_id}.nii.gz',
        'dilated_roi_path': ICH_DILATED_ROI_DIR / f'{case_id}.nii.gz',
    })

missing_prior_paths = []
complete_prior_cases = 0
for group in expected_prior_groups:
    paths = [group['probability_path'], group['binary_path'], group['distance_path'], group['dilated_roi_path']]
    if all(prior_file_ready(path) for path in paths):
        complete_prior_cases += 1
    else:
        missing_prior_paths.extend([path for path in paths if not prior_file_ready(path)])

print('Complete pseudo-ICH prior cases:', complete_prior_cases, '/', len(expected_prior_groups))
if missing_prior_paths:
    print('Missing/corrupt prior files:', len(missing_prior_paths))
    print('First missing/corrupt:', missing_prior_paths[0])

prior_rows = []
if RUN_PSEUDO_ICH_PRIOR_GENERATION and (OVERWRITE_PSEUDO_ICH_PRIORS or missing_prior_paths):
    for row in split_df.itertuples(index=False):
        case_id = str(row.case_id)
        img_path = Path(row.img_path)
        img = sitk.ReadImage(str(img_path))
        ct = sitk.GetArrayFromImage(img)
        spacing_xyz = np.array(img.GetSpacing(), dtype=float)

        prob, binary, components = generate_pseudo_ich_from_ct(ct, spacing_xyz)
        distance_prior = make_distance_prior(binary, spacing_xyz, scale_mm=DISTANCE_SCALE_MM)
        dilated_roi = make_dilated_roi(binary, spacing_xyz, radius_mm=DILATED_ROI_RADIUS_MM)

        prob_path = ICH_PRIOR_PROB_DIR / f'{case_id}.nii.gz'
        binary_path = ICH_PRIOR_MASK_DIR / f'{case_id}.nii.gz'
        distance_path = ICH_DISTANCE_PRIOR_DIR / f'{case_id}.nii.gz'
        dilated_path = ICH_DILATED_ROI_DIR / f'{case_id}.nii.gz'
        write_array_like_reference(prob, img_path, prob_path, dtype=np.float32)
        write_array_like_reference(binary, img_path, binary_path, dtype=np.uint8)
        write_array_like_reference(distance_prior, img_path, distance_path, dtype=np.float32)
        write_array_like_reference(dilated_roi, img_path, dilated_path, dtype=np.uint8)

        prior_rows.append({
            'case_id': case_id,
            'split': row.split,
            'source': PSEUDO_ICH_SOURCE,
            'generation_policy': PSEUDO_ICH_POLICY,
            'probability_path': str(prob_path),
            'binary_path': str(binary_path),
            'distance_path': str(distance_path),
            'dilated_roi_path': str(dilated_path),
            'prob_min': float(prob.min()),
            'prob_max': float(prob.max()),
            'prob_mean': float(prob.mean()),
            'binary_voxels': int(binary.sum()),
            'dilated_voxels': int(dilated_roi.sum()),
            'component_count_kept': int(len(components)),
            'component_volumes_ml': '|'.join(f'{c["volume_ml"]:.4f}' for c in components),
        })
elif RUN_PSEUDO_ICH_PRIOR_GENERATION:
    print('All pseudo-ICH priors already exist. Set OVERWRITE_PSEUDO_ICH_PRIORS=True to regenerate.')
else:
    print('Skipped pseudo-ICH prior generation. Set RUN_PSEUDO_ICH_PRIOR_GENERATION=True to run.')

for row in split_df.itertuples(index=False):
    case_id = str(row.case_id)
    prob_path = ICH_PRIOR_PROB_DIR / f'{case_id}.nii.gz'
    binary_path = ICH_PRIOR_MASK_DIR / f'{case_id}.nii.gz'
    distance_path = ICH_DISTANCE_PRIOR_DIR / f'{case_id}.nii.gz'
    dilated_path = ICH_DILATED_ROI_DIR / f'{case_id}.nii.gz'
    for required_path in [prob_path, binary_path, distance_path, dilated_path]:
        if not prior_file_ready(required_path):
            raise FileNotFoundError(f'Missing or corrupt pseudo-ICH prior for {case_id}: {required_path}')
    prob = sitk.GetArrayFromImage(sitk.ReadImage(str(prob_path)))
    binary = sitk.GetArrayFromImage(sitk.ReadImage(str(binary_path))) > 0
    dilated_roi = sitk.GetArrayFromImage(sitk.ReadImage(str(dilated_path))) > 0
    prior_rows.append({
        'case_id': case_id,
        'split': row.split,
        'source': PSEUDO_ICH_SOURCE,
        'generation_policy': PSEUDO_ICH_POLICY,
        'probability_path': str(prob_path),
        'binary_path': str(binary_path),
        'distance_path': str(distance_path),
        'dilated_roi_path': str(dilated_path),
        'prob_min': float(prob.min()),
        'prob_max': float(prob.max()),
        'prob_mean': float(prob.mean()),
        'binary_voxels': int(binary.sum()),
        'dilated_voxels': int(dilated_roi.sum()),
    })

prior_df = pd.DataFrame(prior_rows).drop_duplicates('case_id', keep='last').sort_values('case_id')
prior_manifest = PSEUDO_ICH_OUTPUT_ROOT / '02_19_pseudo_ich_prior_manifest.csv'
prior_df.to_csv(prior_manifest, index=False)

print('Saved pseudo-ICH prior manifest:', prior_manifest)
print('Pseudo-ICH binary voxels summary:')
display(prior_df[['case_id', 'split', 'binary_voxels', 'dilated_voxels', 'prob_mean']].describe(include='all'))
display(prior_df.head())

## 3. Quick inspect pseudo-ICH prior maps


In [ ]:
INSPECT_PRIORS = True
MAX_INSPECT = 8

if INSPECT_PRIORS:
    inspect_rows = []
    for row in split_df.head(MAX_INSPECT).itertuples(index=False):
        case_id = str(row.case_id)
        paths = {
            'probability': ICH_PRIOR_PROB_DIR / f'{case_id}.nii.gz',
            'binary': ICH_PRIOR_MASK_DIR / f'{case_id}.nii.gz',
            'distance': ICH_DISTANCE_PRIOR_DIR / f'{case_id}.nii.gz',
            'dilated_roi': ICH_DILATED_ROI_DIR / f'{case_id}.nii.gz',
        }
        record = {'case_id': case_id, 'split': row.split}
        for name, path in paths.items():
            if path.exists():
                arr = sitk.GetArrayFromImage(sitk.ReadImage(str(path)))
                record[f'{name}_shape_zyx'] = tuple(arr.shape)
                record[f'{name}_min'] = float(np.min(arr))
                record[f'{name}_max'] = float(np.max(arr))
                record[f'{name}_voxels_gt0'] = int((arr > 0).sum())
            else:
                record[f'{name}_status'] = 'missing'
        inspect_rows.append(record)
    display(pd.DataFrame(inspect_rows))
else:
    print('Skipped prior inspection.')


## 3b. QC visualization for pseudo-ICH priors

PHE masks are shown only for visual sanity checking. They are not used to generate pseudo-ICH maps.


In [ ]:
RUN_PRIOR_QC_FIGURES = True
N_PRIOR_QC_PER_BUCKET = 4

CT_WINDOWS = {
    'brain': (0, 80),
    'blood': (-50, 150),
    'wide': (-100, 300),
}

def window_ct(arr, low=-50, high=150):
    arr = np.asarray(arr, dtype=np.float32)
    arr = np.clip(arr, low, high)
    return (arr - low) / max(high - low, 1e-6)

def largest_slice(mask):
    if not mask.any():
        return mask.shape[0] // 2
    areas = mask.reshape(mask.shape[0], -1).sum(axis=1)
    return int(np.argmax(areas))

def select_prior_qc_cases(prior_df, n_each=4):
    buckets = []
    buckets.append(prior_df.sort_values('binary_voxels').head(n_each).assign(qc_reason='small_prior'))
    buckets.append(prior_df.sort_values('binary_voxels', ascending=False).head(n_each).assign(qc_reason='large_prior'))
    test_df = prior_df[prior_df['split'] == 'test']
    if len(test_df):
        buckets.append(test_df.head(n_each).assign(qc_reason='test_cases'))
    out = pd.concat(buckets, ignore_index=True)
    return out.drop_duplicates('case_id', keep='first')

if RUN_PRIOR_QC_FIGURES:
    import matplotlib.pyplot as plt
    qc_df = select_prior_qc_cases(prior_df, N_PRIOR_QC_PER_BUCKET)
    qc_df.to_csv(PSEUDO_ICH_OUTPUT_ROOT / '02_19_pseudo_prior_qc_selected_cases.csv', index=False)
    for row in qc_df.itertuples(index=False):
        case_id = str(row.case_id)
        split_row = split_df.loc[split_df['case_id'].astype(str) == case_id].iloc[0]
        ct = sitk.GetArrayFromImage(sitk.ReadImage(str(split_row.img_path)))
        phe = sitk.GetArrayFromImage(sitk.ReadImage(str(split_row.phe_mask_path))) > 0
        prob = sitk.GetArrayFromImage(sitk.ReadImage(str(ICH_PRIOR_PROB_DIR / f'{case_id}.nii.gz')))
        binary = sitk.GetArrayFromImage(sitk.ReadImage(str(ICH_PRIOR_MASK_DIR / f'{case_id}.nii.gz'))) > 0
        distance = sitk.GetArrayFromImage(sitk.ReadImage(str(ICH_DISTANCE_PRIOR_DIR / f'{case_id}.nii.gz')))
        dilated = sitk.GetArrayFromImage(sitk.ReadImage(str(ICH_DILATED_ROI_DIR / f'{case_id}.nii.gz'))) > 0
        z = largest_slice(phe | binary | dilated)
        ct_blood = window_ct(ct[z], *CT_WINDOWS['blood'])
        ct_brain = window_ct(ct[z], *CT_WINDOWS['brain'])
        fig, axes = plt.subplots(1, 5, figsize=(18, 4))
        axes[0].imshow(ct_brain, cmap='gray')
        axes[0].set_title(f'{case_id}\n{row.qc_reason}\nCT brain')
        axes[1].imshow(ct_blood, cmap='gray')
        axes[1].imshow(np.ma.masked_where(~phe[z], phe[z]), cmap='Greens', alpha=0.55)
        axes[1].set_title('PHE GT')
        axes[2].imshow(ct_blood, cmap='gray')
        axes[2].imshow(prob[z], cmap='magma', alpha=0.60, vmin=0, vmax=1)
        axes[2].set_title('pseudo-ICH probability')
        axes[3].imshow(distance[z], cmap='viridis', vmin=0, vmax=1)
        axes[3].set_title('distance prior')
        axes[4].imshow(ct_blood, cmap='gray')
        axes[4].imshow(np.ma.masked_where(~dilated[z], dilated[z]), cmap='Blues', alpha=0.25)
        axes[4].imshow(np.ma.masked_where(~phe[z], phe[z]), cmap='Greens', alpha=0.45)
        axes[4].imshow(np.ma.masked_where(~binary[z], binary[z]), cmap='Reds', alpha=0.45)
        axes[4].set_title('PHE + pseudo-ICH + ROI')
        for ax in axes:
            ax.axis('off')
        fig.tight_layout()
        out = PRIOR_QC_DIR / f'{case_id}_{row.qc_reason}_prior_qc.png'
        fig.savefig(out, dpi=160, bbox_inches='tight')
        plt.close(fig)
    print('Saved pseudo-prior QC figures:', PRIOR_QC_DIR)
else:
    print('Skipped prior QC figures.')


## 4. Convert PHE-SICH to selectable multi-channel nnU-Net raw dataset

The conversion is intentionally identical to 02_14b except that prior channels come from CT-only pseudo-ICH maps.


In [ ]:
OVERWRITE_RAW_DATASET = True

PRIOR_CHANNEL_PATHS = {
    'ich_binary': ICH_PRIOR_MASK_DIR,
    'ich_probability': ICH_PRIOR_PROB_DIR,
    'ich_distance': ICH_DISTANCE_PRIOR_DIR,
    'ich_dilated_roi': ICH_DILATED_ROI_DIR,
}
PRIOR_CHANNEL_DISPLAY_NAMES = {
    'ich_binary': 'ICH_binary_prior',
    'ich_probability': 'ICH_probability_prior',
    'ich_distance': 'ICH_distance_prior',
    'ich_dilated_roi': 'ICH_dilated_roi_prior',
}

def write_mask_like_reference(src_path: Path, reference_path: Path, dst_path: Path):
    ref = sitk.ReadImage(str(reference_path))
    seg = sitk.ReadImage(str(src_path))
    arr = sitk.GetArrayFromImage(seg)
    out = sitk.GetImageFromArray((arr > 0).astype(np.uint8))
    if ref.GetSize() != out.GetSize():
        raise ValueError(f'Size mismatch: reference {reference_path} {ref.GetSize()} vs {src_path} {out.GetSize()}')
    out.CopyInformation(ref)
    sitk.WriteImage(out, str(dst_path))

def write_prior_like_reference(src_path: Path, reference_path: Path, dst_path: Path, prior_name: str):
    ref = sitk.ReadImage(str(reference_path))
    src = sitk.ReadImage(str(src_path))
    arr = sitk.GetArrayFromImage(src)
    if prior_name in {'ich_binary', 'ich_dilated_roi'}:
        arr = (arr > 0).astype(np.uint8)
        dtype = np.uint8
    else:
        arr = np.clip(arr.astype(np.float32), 0.0, 1.0)
        dtype = np.float32
    out = sitk.GetImageFromArray(arr.astype(dtype))
    if ref.GetSize() != out.GetSize():
        raise ValueError(f'Size mismatch: reference {reference_path} {ref.GetSize()} vs {src_path} {out.GetSize()}')
    out.CopyInformation(ref)
    sitk.WriteImage(out, str(dst_path))
    return float(arr.min()), float(arr.max()), int((arr > 0).sum())

if DATASET_DIR.exists() and OVERWRITE_RAW_DATASET:
    shutil.rmtree(DATASET_DIR)

imagesTr = DATASET_DIR / 'imagesTr'
labelsTr = DATASET_DIR / 'labelsTr'
imagesTs = DATASET_DIR / 'imagesTs'
labelsTs = DATASET_DIR / 'labelsTs'
for p in [imagesTr, labelsTr, imagesTs, labelsTs]:
    p.mkdir(parents=True, exist_ok=True)

records = []
for row in split_df.itertuples(index=False):
    case_id = str(row.case_id)
    img_src = Path(row.img_path)
    phe_src = Path(row.phe_mask_path)

    if row.split == 'test':
        image_dir = imagesTs
        label_dir = labelsTs
    else:
        image_dir = imagesTr
        label_dir = labelsTr

    img_dst = image_dir / f'{case_id}_0000.nii.gz'
    seg_dst = label_dir / f'{case_id}.nii.gz'
    shutil.copy2(img_src, img_dst)

    channel_record = {}
    for channel_offset, prior_name in enumerate(PRIOR_CHANNELS, start=1):
        prior_dir = PRIOR_CHANNEL_PATHS[prior_name]
        prior_src = prior_dir / f'{case_id}.nii.gz'
        assert prior_src.exists(), f'Missing {prior_name} for {case_id}: {prior_src}'
        prior_dst = image_dir / f'{case_id}_{channel_offset:04d}.nii.gz'
        prior_min, prior_max, prior_voxels = write_prior_like_reference(prior_src, img_dst, prior_dst, prior_name)
        channel_record[f'{prior_name}_channel'] = str(prior_dst)
        channel_record[f'{prior_name}_min'] = prior_min
        channel_record[f'{prior_name}_max'] = prior_max
        channel_record[f'{prior_name}_voxels_gt0'] = prior_voxels

    write_mask_like_reference(phe_src, img_dst, seg_dst)
    records.append({
        'case_id': case_id,
        'split': row.split,
        'ct_channel': str(img_dst),
        'phe_label': str(seg_dst),
        'source_image': str(img_src),
        'source_phe_mask': str(phe_src),
        **channel_record,
    })

channel_names = {'0': 'CT'}
for channel_offset, prior_name in enumerate(PRIOR_CHANNELS, start=1):
    channel_names[str(channel_offset)] = PRIOR_CHANNEL_DISPLAY_NAMES[prior_name]

dataset_json = {
    'channel_names': channel_names,
    'labels': {'background': 0, 'PHE': 1},
    'numTraining': int((split_df['split'] != 'test').sum()),
    'file_ending': '.nii.gz',
    'overwrite_image_reader_writer': 'SimpleITKIO',
}
with open(DATASET_DIR / 'dataset.json', 'w', encoding='utf-8') as f:
    json.dump(dataset_json, f, indent=2)

conversion_df = pd.DataFrame(records)
conversion_manifest = OUTPUT_ROOT / f'02_19_{EXPERIMENT_ID}_conversion_manifest.csv'
conversion_df.to_csv(conversion_manifest, index=False)

print('Experiment:', EXPERIMENT_ID)
print('Prior channels:', PRIOR_CHANNELS)
print('Dataset dir:', DATASET_DIR)
print('imagesTr channel files:', len(list(imagesTr.glob('*.nii.gz'))))
print('labelsTr:', len(list(labelsTr.glob('*.nii.gz'))))
print('imagesTs channel files:', len(list(imagesTs.glob('*.nii.gz'))))
print('labelsTs:', len(list(labelsTs.glob('*.nii.gz'))))
print('dataset.json:', DATASET_DIR / 'dataset.json')
print('conversion manifest:', conversion_manifest)
display(conversion_df.groupby('split').size().rename('cases').reset_index())
display(pd.DataFrame([dataset_json['channel_names']]).T.rename(columns={0: 'channel_name'}))


## 5. Helper call entrypoints



In [ ]:
def call_entrypoint(entrypoint_func, argv):
    old_argv = sys.argv[:]
    sys.argv = list(argv)
    print('>>>', ' '.join(map(str, sys.argv)))
    t0 = time.time()
    try:
        return entrypoint_func()
    finally:
        sys.argv = old_argv
        print(f'Done in {(time.time() - t0) / 60:.1f} min')


## 6. Plan and preprocess selected Dataset



In [ ]:
CONFIGURATION = '3d_fullres'
RUN_PLAN_PREPROCESS = True

if RUN_PLAN_PREPROCESS:
    from nnunetv2.experiment_planning.plan_and_preprocess_entrypoints import plan_and_preprocess_entry
    old_n_proc_da = os.environ.get('nnUNet_n_proc_DA')
    os.environ['nnUNet_n_proc_DA'] = '1'
    try:
        call_entrypoint(plan_and_preprocess_entry, [
            'nnUNetv2_plan_and_preprocess',
            '-d', str(DATASET_ID),
            '-c', CONFIGURATION,
            '--verify_dataset_integrity',
            '-npfp', '2',
            '-np', '1',
        ])
    finally:
        os.environ['nnUNet_n_proc_DA'] = old_n_proc_da if old_n_proc_da is not None else '0'
else:
    print('Skipped. Set RUN_PLAN_PREPROCESS=True to run.')


## 7. Write manual fold-0 split



In [ ]:
preprocessed_dataset_dir = NNUNET_PREPROCESSED / DATASET_NAME
split_json_path = preprocessed_dataset_dir / 'splits_final.json'

train_ids = split_df.loc[split_df['split'] == 'train', 'case_id'].astype(str).tolist()
val_ids = split_df.loc[split_df['split'] == 'val', 'case_id'].astype(str).tolist()
test_ids = split_df.loc[split_df['split'] == 'test', 'case_id'].astype(str).tolist()

manual_splits = [{'train': train_ids, 'val': val_ids}]

if preprocessed_dataset_dir.exists():
    with open(split_json_path, 'w', encoding='utf-8') as f:
        json.dump(manual_splits, f, indent=2)
    print('Wrote:', split_json_path)
    print('train:', len(train_ids), 'val:', len(val_ids), 'test:', len(test_ids))
else:
    print('Preprocessed folder not found yet:', preprocessed_dataset_dir)
    print('Run plan/preprocess first, then rerun this cell.')


## 8. Train PHE model with selected ICH prior channels

This trains a new PHE model for the current `EXPERIMENT_ID`. The pseudo-ICH remains frozen.


In [ ]:
RUN_TRAIN = True
CONFIGURATION = globals().get('CONFIGURATION', '3d_fullres')
FOLD = 0
TRAINER = 'nnUNetTrainer_250epochs'
PLANS = 'nnUNetPlans'
EXPORT_VAL_NPZ = True
CONTINUE_TRAINING = True

MODEL_DIR = NNUNET_RESULTS / DATASET_NAME / f'{TRAINER}__{PLANS}__{CONFIGURATION}'
FOLD_DIR = MODEL_DIR / f'fold_{FOLD}'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FOLD_DIR.mkdir(parents=True, exist_ok=True)

EFFECTIVE_CONTINUE_TRAINING = CONTINUE_TRAINING
if CONTINUE_TRAINING:
    available_checkpoints = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth'))
    if not available_checkpoints:
        print('CONTINUE_TRAINING=True but no checkpoint exists yet; starting a fresh training run.')
        EFFECTIVE_CONTINUE_TRAINING = False
    else:
        print('Continuing from available checkpoints:', available_checkpoints)

if RUN_TRAIN:
    import torch
    from nnunetv2.run.run_training import run_training
    os.environ['nnUNet_n_proc_DA'] = '0'
    os.environ['nnUNet_compile'] = 'False'
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Training device:', device)
    print('Model dir:', MODEL_DIR)
    print('Fold dir :', FOLD_DIR)
    print('nnUNet_n_proc_DA:', os.environ.get('nnUNet_n_proc_DA'))
    print('nnUNet_compile:', os.environ.get('nnUNet_compile'))
    print('continue_training:', EFFECTIVE_CONTINUE_TRAINING)
    # Windows/Jupyter guard: some nnU-Net builds try to copy dataset_fingerprint.json
    # before recreating the trainer output folder. Ensure the destination parent exists.
    import shutil as _shutil
    if not hasattr(_shutil, '_codex_original_copyfile'):
        _shutil._codex_original_copyfile = _shutil.copyfile

    def _copyfile_with_parent(src, dst, *args, **kwargs):
        if str(src).endswith('dataset_fingerprint.json'):
            Path(dst).parent.mkdir(parents=True, exist_ok=True)
        return _shutil._codex_original_copyfile(src, dst, *args, **kwargs)

    _shutil.copyfile = _copyfile_with_parent
    try:
        run_training(
            str(DATASET_ID),
            CONFIGURATION,
            str(FOLD),
            trainer_class_name=TRAINER,
            plans_identifier=PLANS,
            export_validation_probabilities=EXPORT_VAL_NPZ,
            continue_training=EFFECTIVE_CONTINUE_TRAINING,
            device=device,
        )
    finally:
        _shutil.copyfile = _shutil._codex_original_copyfile
else:
    print('Skipped. Set RUN_TRAIN=True to train.')


## 9. Predict test set and evaluate PHE



In [ ]:
RUN_PHE_PREDICT = True
RUN_PHE_EVALUATE = True
DISABLE_TTA = False
CHECKPOINT = 'auto'

imagesTs = DATASET_DIR / 'imagesTs'
labelsTs = DATASET_DIR / 'labelsTs'
MODEL_DIR = NNUNET_RESULTS / DATASET_NAME / f'{TRAINER}__{PLANS}__{CONFIGURATION}'
FOLD_DIR = MODEL_DIR / f'fold_{FOLD}'

def resolve_checkpoint_name(checkpoint_setting='auto'):
    if checkpoint_setting != 'auto':
        ckpt = FOLD_DIR / checkpoint_setting
        if not ckpt.exists():
            available = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth')) if FOLD_DIR.exists() else []
            raise FileNotFoundError(f'Missing checkpoint: {ckpt}. Available: {available}')
        return checkpoint_setting
    for name in ['checkpoint_best.pth', 'checkpoint_final.pth', 'checkpoint_latest.pth']:
        if (FOLD_DIR / name).exists():
            return name
    available = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth')) if FOLD_DIR.exists() else []
    raise FileNotFoundError(f'No checkpoint found in {FOLD_DIR}. Available: {available}')

RESOLVED_CHECKPOINT = resolve_checkpoint_name(CHECKPOINT)
PRED_DIR = OUTPUT_ROOT / f'predictions_{CONFIGURATION}_{TRAINER}_fold{FOLD}_{RESOLVED_CHECKPOINT.replace(".pth", "")}'
SUMMARY_JSON = PRED_DIR / 'summary.json'

print('Model dir        :', MODEL_DIR)
print('Fold dir         :', FOLD_DIR)
print('Using checkpoint :', RESOLVED_CHECKPOINT)

if RUN_PHE_PREDICT:
    import torch
    from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('PHE predict device:', device)
    predictor = nnUNetPredictor(
        tile_step_size=0.5,
        use_gaussian=True,
        use_mirroring=not DISABLE_TTA,
        perform_everything_on_device=(device.type == 'cuda'),
        device=device,
        verbose=False,
        verbose_preprocessing=False,
        allow_tqdm=True,
    )
    predictor.initialize_from_trained_model_folder(
        str(MODEL_DIR),
        use_folds=(FOLD,),
        checkpoint_name=RESOLVED_CHECKPOINT,
    )
    predictor.predict_from_files(
        str(imagesTs),
        str(PRED_DIR),
        save_probabilities=False,
        overwrite=True,
        num_processes_preprocessing=1,
        num_processes_segmentation_export=1,
        folder_with_segs_from_prev_stage=None,
        num_parts=1,
        part_id=0,
    )
else:
    print('Skipped prediction. Set RUN_PHE_PREDICT=True to run inference.')

if RUN_PHE_EVALUATE:
    from nnunetv2.evaluation.evaluate_predictions import evaluate_folder_entry_point
    plans_file = NNUNET_PREPROCESSED / DATASET_NAME / f'{PLANS}.json'
    dataset_json_file = DATASET_DIR / 'dataset.json'
    call_entrypoint(evaluate_folder_entry_point, [
        'nnUNetv2_evaluate_folder',
        str(labelsTs),
        str(PRED_DIR),
        '-djfile', str(dataset_json_file),
        '-pfile', str(plans_file),
        '-o', str(SUMMARY_JSON),
        '-np', '2',
    ])
else:
    print('Skipped evaluation. Set RUN_PHE_EVALUATE=True after prediction.')

print('Prediction dir:', PRED_DIR)
print('Summary json  :', SUMMARY_JSON)


## 9b. Pseudo-ICH-guided post-processing


In [ ]:
RUN_POSTPROCESS = True
POSTPROCESS_MIN_SIZE_VOXELS = 50
POSTPROCESS_ROI_RADIUS_MM = 25.0

POSTPROCESS_PRED_DIR = OUTPUT_ROOT / f'postprocessed_{CONFIGURATION}_{TRAINER}_fold{FOLD}_{RESOLVED_CHECKPOINT.replace(".pth", "")}'
POSTPROCESS_SUMMARY_JSON = POSTPROCESS_PRED_DIR / 'summary.json'

def make_ich_roi(binary_ich, spacing_xyz, radius_mm):
    binary_ich = binary_ich.astype(bool)
    if not binary_ich.any():
        return np.ones(binary_ich.shape, dtype=bool)
    spacing_zyx = tuple(float(x) for x in spacing_xyz[::-1])
    dist_mm = ndi.distance_transform_edt(~binary_ich, sampling=spacing_zyx)
    return dist_mm <= float(radius_mm)

def postprocess_phe_with_ich(phe_pred, ich_prior, spacing_xyz, min_size=50, radius_mm=25.0):
    phe_pred = phe_pred.astype(bool)
    ich_prior = ich_prior.astype(bool)
    ich_roi = make_ich_roi(ich_prior, spacing_xyz, radius_mm)
    labeled, num = ndi.label(phe_pred)
    output = np.zeros_like(phe_pred, dtype=np.uint8)
    kept = 0
    removed_small = 0
    removed_far = 0
    for comp_id in range(1, num + 1):
        comp = labeled == comp_id
        comp_size = int(comp.sum())
        if comp_size < int(min_size):
            removed_small += 1
            continue
        if np.logical_and(comp, ich_roi).sum() == 0:
            removed_far += 1
            continue
        output[comp] = 1
        kept += 1
    return output, {'components': int(num), 'kept': kept, 'removed_small': removed_small, 'removed_far': removed_far}

def write_mask_array_like_reference(mask_arr, reference_path: Path, out_path: Path):
    ref = sitk.ReadImage(str(reference_path))
    out = sitk.GetImageFromArray(mask_arr.astype(np.uint8))
    if out.GetSize() != ref.GetSize():
        raise ValueError(f'Size mismatch for {out_path}: {out.GetSize()} vs ref {ref.GetSize()}')
    out.CopyInformation(ref)
    sitk.WriteImage(out, str(out_path))

if RUN_POSTPROCESS:
    POSTPROCESS_PRED_DIR.mkdir(parents=True, exist_ok=True)
    pp_rows = []
    for case_id in split_df.loc[split_df['split'] == 'test', 'case_id'].astype(str):
        raw_pred_path = PRED_DIR / f'{case_id}.nii.gz'
        ich_prior_path = ICH_PRIOR_MASK_DIR / f'{case_id}.nii.gz'
        out_path = POSTPROCESS_PRED_DIR / f'{case_id}.nii.gz'
        if not raw_pred_path.exists():
            print('Missing raw PHE prediction:', raw_pred_path)
            continue
        raw_img = sitk.ReadImage(str(raw_pred_path))
        raw_pred = sitk.GetArrayFromImage(raw_img) > 0
        ich_prior = sitk.GetArrayFromImage(sitk.ReadImage(str(ich_prior_path))) > 0
        spacing_xyz = np.array(raw_img.GetSpacing(), dtype=float)
        pp_pred, stats = postprocess_phe_with_ich(
            raw_pred,
            ich_prior,
            spacing_xyz,
            min_size=POSTPROCESS_MIN_SIZE_VOXELS,
            radius_mm=POSTPROCESS_ROI_RADIUS_MM,
        )
        write_mask_array_like_reference(pp_pred, raw_pred_path, out_path)
        pp_rows.append({
            'case_id': case_id,
            'raw_voxels': int(raw_pred.sum()),
            'post_voxels': int(pp_pred.sum()),
            **stats,
        })
    postprocess_df = pd.DataFrame(pp_rows)
    postprocess_manifest = OUTPUT_ROOT / f'02_19_{EXPERIMENT_ID}_postprocess_manifest.csv'
    postprocess_df.to_csv(postprocess_manifest, index=False)
    print('Saved postprocessed predictions:', POSTPROCESS_PRED_DIR)
    print('Saved postprocess manifest:', postprocess_manifest)
    display(postprocess_df)

    if RUN_PHE_EVALUATE and len(postprocess_df):
        plans_file = NNUNET_PREPROCESSED / DATASET_NAME / f'{PLANS}.json'
        dataset_json_file = DATASET_DIR / 'dataset.json'
        call_entrypoint(evaluate_folder_entry_point, [
            'nnUNetv2_evaluate_folder',
            str(labelsTs),
            str(POSTPROCESS_PRED_DIR),
            '-djfile', str(dataset_json_file),
            '-pfile', str(plans_file),
            '-o', str(POSTPROCESS_SUMMARY_JSON),
            '-np', '2',
        ])
else:
    print('Skipped pseudo-ICH-guided post-processing.')

## 10. Export compact metrics CSV


In [ ]:
def strip_nii_gz_name(path_like):
    name = Path(str(path_like)).name
    return name[:-7] if name.endswith('.nii.gz') else Path(name).stem

def export_summary(summary_json: Path, source_name: str):
    if not summary_json.exists():
        print('No summary yet:', summary_json)
        return None, None
    with open(summary_json, 'r', encoding='utf-8') as f:
        summary = json.load(f)

    rows = []
    for item in summary.get('metric_per_case', []):
        metrics = item.get('metrics', {}).get('1', {})
        rows.append({
            'experiment_id': EXPERIMENT_ID,
            'source': source_name,
            'case_id': strip_nii_gz_name(item.get('prediction_file', '')),
            'prediction_file': item.get('prediction_file', ''),
            'reference_file': item.get('reference_file', ''),
            **metrics,
        })
    per_case_df = pd.DataFrame(rows)

    summary_rows = []
    numeric_cols = [c for c in per_case_df.columns if c not in {'experiment_id', 'source', 'case_id', 'prediction_file', 'reference_file'}]
    for col in numeric_cols:
        vals = pd.to_numeric(per_case_df[col], errors='coerce').dropna()
        if len(vals) == 0:
            continue
        summary_rows.append({
            'experiment_id': EXPERIMENT_ID,
            'source': source_name,
            'label': 'PHE',
            'metric': col,
            'mean': float(vals.mean()),
            'median': float(vals.median()),
            'std': float(vals.std(ddof=0)),
            'n': int(len(vals)),
        })
    summary_df = pd.DataFrame(summary_rows)
    return per_case_df, summary_df

raw_per_case_df, raw_summary_df = export_summary(SUMMARY_JSON, 'raw')
post_per_case_df, post_summary_df = export_summary(globals().get('POSTPROCESS_SUMMARY_JSON', Path('__missing__')), 'postprocessed')

per_case_frames = [df for df in [raw_per_case_df, post_per_case_df] if df is not None]
summary_frames = [df for df in [raw_summary_df, post_summary_df] if df is not None]

if per_case_frames:
    metrics_df = pd.concat(per_case_frames, ignore_index=True)
    metrics_csv = OUTPUT_ROOT / f'02_19_{EXPERIMENT_ID}_metrics_per_case_raw_vs_post.csv'
    metrics_df.to_csv(metrics_csv, index=False)
    print('Wrote:', metrics_csv)
else:
    print('No per-case metrics exported.')

if summary_frames:
    summary_df = pd.concat(summary_frames, ignore_index=True)
    summary_csv = OUTPUT_ROOT / f'02_19_{EXPERIMENT_ID}_metrics_summary_raw_vs_post.csv'
    summary_df.to_csv(summary_csv, index=False)
    print('Wrote:', summary_csv)
    display(summary_df.pivot_table(index='metric', columns='source', values='mean'))
else:
    print('No metric summary exported. Run train -> predict -> evaluate first.')

experiment_record = {
    'experiment_id': EXPERIMENT_ID,
    'dataset_id': DATASET_ID,
    'dataset_name': DATASET_NAME,
    'prior_channels': '|'.join(PRIOR_CHANNELS),
    'prior_source': PSEUDO_ICH_SOURCE,
    'prior_policy': PSEUDO_ICH_POLICY,
    'pseudo_soft_low_hu': PSEUDO_ICH_SOFT_LOW_HU,
    'pseudo_core_low_hu': PSEUDO_ICH_CORE_LOW_HU,
    'pseudo_core_high_hu': PSEUDO_ICH_CORE_HIGH_HU,
    'pseudo_soft_high_hu': PSEUDO_ICH_SOFT_HIGH_HU,
    'output_root': str(OUTPUT_ROOT),
}
experiment_summary_csv = OUTPUT_BASE / '02_19_experiment_registry.csv'
existing = pd.read_csv(experiment_summary_csv) if experiment_summary_csv.exists() else pd.DataFrame()
registry_df = pd.concat([existing, pd.DataFrame([experiment_record])], ignore_index=True)
registry_df = registry_df.drop_duplicates(['experiment_id', 'dataset_id'], keep='last')
registry_df.to_csv(experiment_summary_csv, index=False)
print('Updated 02_19 experiment registry:', experiment_summary_csv)


## 11. PHE detection / early detection analysis


In [ ]:
RUN_PHE_DETECTION_ANALYSIS = True

# GT positive threshold should usually stay at 0: any manual PHE voxel means PHE is present.
GT_POSITIVE_VOLUME_THRESHOLD_ML = 0.0

# Main operating point. The threshold sweep below helps decide whether this should change.
PRED_POSITIVE_VOLUME_THRESHOLD_ML = 0.5
DETECTION_THRESHOLD_SWEEP_ML = [0.0, 0.1, 0.25, 0.5, 1.0, 2.0]

# Proxy for early/subtle PHE when onset-time metadata is unavailable.
SMALL_PHE_VOLUME_CUTOFF_ML = 5.0


def load_binary_mask(path: Path):
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img) > 0
    return img, arr


def mask_volume_ml(mask, spacing_xyz):
    voxel_ml = float(np.prod(np.asarray(spacing_xyz, dtype=float)) / 1000.0)
    return float(mask.sum() * voxel_ml)


def safe_div(num, den):
    return np.nan if den == 0 else float(num / den)


def detection_counts(gt_positive, pred_positive):
    gt_positive = np.asarray(gt_positive, dtype=bool)
    pred_positive = np.asarray(pred_positive, dtype=bool)
    tp = int(np.logical_and(gt_positive, pred_positive).sum())
    fp = int(np.logical_and(~gt_positive, pred_positive).sum())
    fn = int(np.logical_and(gt_positive, ~pred_positive).sum())
    tn = int(np.logical_and(~gt_positive, ~pred_positive).sum())
    return tp, fp, fn, tn


def detection_metric_row(df, source, cohort, pred_threshold_ml):
    tp, fp, fn, tn = detection_counts(df['gt_positive'], df['pred_positive'])
    sensitivity = safe_div(tp, tp + fn)
    specificity = safe_div(tn, tn + fp)
    precision = safe_div(tp, tp + fp)
    recall = sensitivity
    npv = safe_div(tn, tn + fn)
    f1 = np.nan if precision != precision or recall != recall or (precision + recall) == 0 else 2 * precision * recall / (precision + recall)
    accuracy = safe_div(tp + tn, tp + fp + fn + tn)
    balanced_accuracy = np.nanmean([sensitivity, specificity])
    return {
        'experiment_id': EXPERIMENT_ID,
        'source': source,
        'cohort': cohort,
        'pred_threshold_ml': float(pred_threshold_ml),
        'gt_threshold_ml': float(GT_POSITIVE_VOLUME_THRESHOLD_ML),
        'small_phe_cutoff_ml': float(SMALL_PHE_VOLUME_CUTOFF_ML),
        'n': int(len(df)),
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'precision': precision,
        'recall': recall,
        'npv': npv,
        'f1': f1,
        'accuracy': accuracy,
        'balanced_accuracy': balanced_accuracy,
        'gt_positive_rate': float(pd.to_numeric(df['gt_positive'], errors='coerce').mean()) if len(df) else np.nan,
        'pred_positive_rate': float(pd.to_numeric(df['pred_positive'], errors='coerce').mean()) if len(df) else np.nan,
        'mean_gt_volume_ml': float(df['gt_volume_ml'].mean()) if len(df) else np.nan,
        'mean_pred_volume_ml': float(df['pred_volume_ml'].mean()) if len(df) else np.nan,
    }


def available_detection_sources():
    sources = []
    if 'PRED_DIR' in globals():
        sources.append(('raw', Path(PRED_DIR)))
    if 'POSTPROCESS_PRED_DIR' in globals() and Path(POSTPROCESS_PRED_DIR).exists():
        sources.append(('postprocessed', Path(POSTPROCESS_PRED_DIR)))
    return sources


if RUN_PHE_DETECTION_ANALYSIS:
    if 'labelsTs' not in globals():
        raise RuntimeError('labelsTs is not defined. Run dataset conversion/prediction cells before this detection analysis.')

    detection_sources = available_detection_sources()
    if not detection_sources:
        print('No prediction folders found. Run PHE prediction first, then rerun this cell.')
    else:
        case_rows = []
        test_case_ids = split_df.loc[split_df['split'] == 'test', 'case_id'].astype(str).tolist()
        for source_name, pred_dir in detection_sources:
            for case_id in test_case_ids:
                gt_path = labelsTs / f'{case_id}.nii.gz'
                pred_path = pred_dir / f'{case_id}.nii.gz'
                row = {
                    'experiment_id': EXPERIMENT_ID,
                    'source': source_name,
                    'case_id': case_id,
                    'gt_path': str(gt_path),
                    'pred_path': str(pred_path),
                    'status': 'ok',
                }
                if not gt_path.exists():
                    row['status'] = 'missing_gt'
                    case_rows.append(row)
                    continue
                if not pred_path.exists():
                    row['status'] = 'missing_prediction'
                    case_rows.append(row)
                    continue

                gt_img, gt_mask = load_binary_mask(gt_path)
                _, pred_mask = load_binary_mask(pred_path)
                if gt_mask.shape != pred_mask.shape:
                    row['status'] = f'shape_mismatch_gt_{gt_mask.shape}_pred_{pred_mask.shape}'
                    case_rows.append(row)
                    continue

                spacing_xyz = np.array(gt_img.GetSpacing(), dtype=float)
                gt_volume_ml = mask_volume_ml(gt_mask, spacing_xyz)
                pred_volume_ml = mask_volume_ml(pred_mask, spacing_xyz)
                gt_positive = gt_volume_ml > GT_POSITIVE_VOLUME_THRESHOLD_ML
                pred_positive = pred_volume_ml >= PRED_POSITIVE_VOLUME_THRESHOLD_ML
                row.update({
                    'gt_volume_ml': gt_volume_ml,
                    'pred_volume_ml': pred_volume_ml,
                    'gt_voxels': int(gt_mask.sum()),
                    'pred_voxels': int(pred_mask.sum()),
                    'gt_positive': bool(gt_positive),
                    'pred_positive': bool(pred_positive),
                    'small_phe_proxy': bool(gt_positive and gt_volume_ml <= SMALL_PHE_VOLUME_CUTOFF_ML),
                    'volume_error_ml': float(pred_volume_ml - gt_volume_ml),
                    'abs_volume_error_ml': float(abs(pred_volume_ml - gt_volume_ml)),
                })
                case_rows.append(row)

        detection_case_df = pd.DataFrame(case_rows)
        detection_cases_csv = OUTPUT_ROOT / f'02_19_{EXPERIMENT_ID}_phe_detection_per_case.csv'
        detection_case_df.to_csv(detection_cases_csv, index=False)
        print('Wrote:', detection_cases_csv)

        valid_df = detection_case_df[detection_case_df['status'] == 'ok'].copy()
        summary_rows = []
        threshold_rows = []
        if len(valid_df):
            for source_name, source_df in valid_df.groupby('source'):
                gt_class_counts = source_df['gt_positive'].value_counts(dropna=False).to_dict()
                print(f'GT detection class counts for {source_name}:', gt_class_counts)
                if source_df['gt_positive'].nunique(dropna=True) < 2:
                    print(f'Warning: {source_name} has only one GT detection class; specificity and balanced accuracy may be limited.')
                summary_rows.append(detection_metric_row(source_df, source_name, 'all_test', PRED_POSITIVE_VOLUME_THRESHOLD_ML))

                small_df = source_df[source_df['small_phe_proxy']]
                if len(small_df):
                    summary_rows.append(detection_metric_row(
                        small_df,
                        source_name,
                        f'small_phe_le_{SMALL_PHE_VOLUME_CUTOFF_ML:g}ml',
                        PRED_POSITIVE_VOLUME_THRESHOLD_ML,
                    ))

                large_df = source_df[source_df['gt_positive'] & ~source_df['small_phe_proxy']]
                if len(large_df):
                    summary_rows.append(detection_metric_row(
                        large_df,
                        source_name,
                        f'larger_phe_gt_{SMALL_PHE_VOLUME_CUTOFF_ML:g}ml',
                        PRED_POSITIVE_VOLUME_THRESHOLD_ML,
                    ))

                for threshold_ml in DETECTION_THRESHOLD_SWEEP_ML:
                    sweep_df = source_df.copy()
                    sweep_df['pred_positive'] = sweep_df['pred_volume_ml'] >= float(threshold_ml)
                    threshold_rows.append(detection_metric_row(sweep_df, source_name, 'threshold_sweep_all_test', threshold_ml))

            detection_summary_df = pd.DataFrame(summary_rows)
            detection_threshold_sweep_df = pd.DataFrame(threshold_rows)
            detection_summary_csv = OUTPUT_ROOT / f'02_19_{EXPERIMENT_ID}_phe_detection_summary.csv'
            detection_threshold_sweep_csv = OUTPUT_ROOT / f'02_19_{EXPERIMENT_ID}_phe_detection_threshold_sweep.csv'
            detection_summary_df.to_csv(detection_summary_csv, index=False)
            detection_threshold_sweep_df.to_csv(detection_threshold_sweep_csv, index=False)
            print('Wrote:', detection_summary_csv)
            print('Wrote:', detection_threshold_sweep_csv)
            display(detection_summary_df[['source', 'cohort', 'n', 'tp', 'fp', 'fn', 'tn', 'sensitivity', 'specificity', 'precision', 'f1', 'accuracy']])
            display(detection_threshold_sweep_df[['source', 'pred_threshold_ml', 'sensitivity', 'specificity', 'precision', 'f1', 'accuracy']])
            display(valid_df.sort_values(['source', 'gt_volume_ml'])[['source', 'case_id', 'gt_volume_ml', 'pred_volume_ml', 'gt_positive', 'pred_positive', 'small_phe_proxy']])
        else:
            print('No valid prediction/GT pairs found for detection analysis.')
else:
    print('Skipped PHE detection analysis.')


## Notes

- `02_19` is the fair pseudo-ICH control for `02_14b`, now configured for local execution.
- It reads local `nnUNet-master` and local `PHE-SICH-CT-IDS`, then writes outputs to `outputs_02_19_nnunet_phe_pseudo_ich_prior_control` and model checkpoints to `o19_results`.
- It keeps the same PHE-SICH split logic, nnU-Net conversion, trainer, prediction, evaluation, post-processing, and detection analysis structure as `02_14b`.
- Difference vs `02_14b`: no `02_13b` ICH teacher checkpoint is loaded. Prior channels are generated directly from CT intensities on PHE-SICH.
- Fairness rule: pseudo-ICH generation uses CT only. PHE masks are used only as PHE segmentation labels/evaluation targets and optional QC overlays.
- Change `EXPERIMENT_ID` in the first code cell to run ablations:
  `phe_pseudo_binary_prior`, `phe_pseudo_probability_prior`, `phe_pseudo_distance_prior`, `phe_pseudo_prob_distance_prior`, or `phe_pseudo_prob_distance_dilated_prior`.
- Compare the default `phe_pseudo_prob_distance_prior` against 14b `phe_prob_distance_prior` on the same test split and same checkpoint policy.
